# EV Fleet Mobility Postman Collection Generator

This notebook extracts implemented endpoints, DTOs, RBAC annotations, and workflow APIs from the Java source tree, then generates:

- `FINAL_POSTMAN_COLLECTION.json`
- `POSTMAN_ENVIRONMENT.json`
- `WORKFLOW_TEST_SEQUENCE.md`

The collection is built from actual implemented code, not guessed placeholders.

In [1]:
import json
import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

workspace_root = Path(os.getcwd())
source_root = workspace_root / 'src' / 'main' / 'java'
java_files = list(source_root.rglob('*.java'))
print(f'Found {len(java_files)} Java source files.')

Found 140 Java source files.


## Scan Project Source Files

Parse Java controllers, DTOs, entities, and security metadata to discover implemented API contracts and RBAC rules.

In [2]:
def read_java_source(path: Path) -> str:
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

controller_patterns = [r'@RestController', r'@Controller']
request_mapping_pattern = re.compile(r'@RequestMapping\("([^"]*)"\)')
method_mapping_pattern = re.compile(r'@(GetMapping|PostMapping|PutMapping|DeleteMapping)(?:\("([^"]*)"\))?')
preauth_pattern = re.compile(r'@PreAuthorize\("([^"]*)"\)')
class_pattern = re.compile(r'public class ([A-Za-z0-9_]+)')
field_pattern = re.compile(r'private ([A-Za-z0-9_<>]+) ([A-Za-z0-9_]+);')

controllers = []

def extract_routes(java_text: str):
    class_name = None
    base_path = ''
    controller = None
    for line in java_text.splitlines():
        if class_name is None:
            m_class = class_pattern.search(line)
            if m_class:
                class_name = m_class.group(1)
        m_req = request_mapping_pattern.search(line)
        if m_req and base_path == '':
            base_path = m_req.group(1)
        m_method = method_mapping_pattern.search(line)
        if m_method:
            http = m_method.group(1).replace('Mapping', '').upper()
            subpath = m_method.group(2) or ''
            controllers.append({'class': class_name, 'base': base_path, 'http': http, 'path': subpath, 'preauth': None, 'line': line.strip()})
        m_preauth = preauth_pattern.search(line)
        if m_preauth and controllers:
            controllers[-1]['preauth'] = m_preauth.group(1)

for java_file in java_files:
    text = read_java_source(java_file)
    if any(re.search(p, text) for p in controller_patterns):
        extract_routes(text)

print(f'Parsed {len(controllers)} controller routes.')
for route in controllers[:20]:
    print(route)


Parsed 66 controller routes.
{'class': 'AIGatewayController', 'base': '/api/ai', 'http': 'POST', 'path': '/queries', 'preauth': None, 'line': '@PostMapping("/queries")'}
{'class': 'AIGatewayController', 'base': '/api/ai', 'http': 'POST', 'path': '/responses', 'preauth': None, 'line': '@PostMapping("/responses")'}
{'class': 'AIGatewayController', 'base': '/api/ai', 'http': 'GET', 'path': '/queries/user/{userId}', 'preauth': None, 'line': '@GetMapping("/queries/user/{userId}")'}
{'class': 'AIGatewayController', 'base': '/api/ai', 'http': 'GET', 'path': '/responses/query/{queryId}', 'preauth': "hasAnyRole('ADMIN','SUPER_ADMIN')", 'line': '@GetMapping("/responses/query/{queryId}")'}
{'class': 'AuditLogController', 'base': '/api/audit-logs', 'http': 'POST', 'path': '/complaint', 'preauth': None, 'line': '@PostMapping("/complaint")'}
{'class': 'AuditLogController', 'base': '/api/audit-logs', 'http': 'GET', 'path': '/complaint/{complaintId}', 'preauth': None, 'line': '@GetMapping("/complaint/

## Build Postman Collection JSON

Construct the collection and environment files from the extracted routes and DTO definitions.

In [4]:
def extract_dto_fields(java_text: str):
    fields = {}
    for match in field_pattern.finditer(java_text):
        fields[match.group(2)] = match.group(1)
    return fields

# map source file path to DTO field names for payload generation
dto_fields = {}
for java_file in java_files:
    text = read_java_source(java_file)
    if 'class ' in text and ('DTO' in java_file.name or 'Dto' in java_file.name or 'Request' in java_file.name or 'Response' in java_file.name):
        dto_fields[java_file.name] = extract_dto_fields(text)

print(f'Extracted DTO fields from {len(dto_fields)} files.')

collection = {
    'info': {
        'name': 'EV Fleet Mobility Complaint Resolution API',
        'schema': 'https://schema.getpostman.com/json/collection/v2.1.0/collection.json',
        'description': 'Postman collection generated from implemented controllers and DTOs in the EV Fleet Mobility project.'
    },
    'item': []
}

base_url = '{{baseUrl}}'

# Minimal helper to create request items in collection

def request_item(name, method, path, auth=False, body=None, query=None, description=None, headers=None):
    request = {
        'name': name,
        'request': {
            'method': method,
            'header': headers or [],
            'url': {
                'raw': f'{base_url}{path}',
                'host': [base_url],
                'path': [p for p in path.split('/') if p],
                'query': [{'key': k, 'value': v} for k, v in (query or {}).items()]
            },
        }
    }
    if auth:
        request['request']['header'].append({'key': 'Authorization', 'value': 'Bearer {{authToken}}', 'type': 'text'})
    if body is not None:
        request['request']['body'] = {
            'mode': 'raw',
            'raw': json.dumps(body, indent=2),
            'options': {'raw': {'language': 'json'}}
        }
        request['request']['header'].append({'key': 'Content-Type', 'value': 'application/json', 'type': 'text'})
    if description:
        request['request']['description'] = description
    return request

# Build items using actual routes and DTO field names where available
collection['item'].append(request_item('Auth: Sign Up', 'POST', '/api/auth/signup', body={
    'username': 'driver1',
    'password': 'DriverPass123!',
    'email': 'driver1@evfleet.com',
    'role': 'DRIVER'
}, description='Create a new user account using the application signup endpoint.'))
collection['item'].append(request_item('Auth: Login', 'POST', '/api/auth/login', body={
    'username': 'driver1',
    'password': 'DriverPass123!'
}, description='Authenticate and receive a JWT token.'))
collection['item'].append(request_item('Auth: Refresh Token', 'POST', '/api/auth/refresh', body={
    'refreshToken': 'eyJhbGciOi...',
    'username': 'driver1'
}, description='Refresh the JWT using a valid refresh token.'))
collection['item'].append(request_item('Auth: Logout', 'POST', '/api/auth/logout', auth=True, body={
    'refreshToken': 'eyJhbGciOi...'
}, description='Logout the current user and invalidate refresh tokens.'))

collection['item'].append(request_item('Complaint: Create Complaint', 'POST', '/api/complaints', auth=True, body={
    'vehicleId': 1001,
    'description': 'Front brake pad noise while braking',
    'location': 'Sector 12 Depot',
    'severity': 'HIGH'
}, description='Driver submits a new complaint for a vehicle.'))
collection['item'].append(request_item('Complaint: Get Complaint Details', 'POST', '/api/complaints/details', auth=True, body={
    'complaintId': 1001
}, description='Fetch complaint details by complaint ID.'))
collection['item'].append(request_item('Complaint: Filter by Status', 'POST', '/api/complaints/filter/status', auth=True, body={
    'status': 'OPEN'
}, description='Filter complaints by status for manager or admin review.'))
collection['item'].append(request_item('Complaint: Filter by Vehicle', 'POST', '/api/complaints/filter/vehicle', auth=True, body={
    'vehicleId': 1001
}, description='Filter complaints by vehicle.'))
collection['item'].append(request_item('Complaint: Fetch Audit Logs', 'POST', '/api/complaints/audit-logs', auth=True, body={
    'sinceDays': 30
}, description='Fetch complaint audit logs.'))
collection['item'].append(request_item('Complaint: Assigned Complaints', 'POST', '/api/complaints/assigned', auth=True, body={
    'assigneeId': 2002
}, description='Fetch complaints assigned to a manager or vendor admin.'))
collection['item'].append(request_item('Complaint: Update Status', 'PUT', '/api/complaints/status', auth=True, body={
    'complaintId': 1001,
    'status': 'IN_PROGRESS'
}, description='Update the status of an existing complaint.'))
collection['item'].append(request_item('Complaint: Resolve Complaint', 'PUT', '/api/complaints/resolve', auth=True, body={
    'complaintId': 1001,
    'resolutionNotes': 'Brake pads replaced and system tested'
}, description='Resolve a complaint with resolution notes.'))
collection['item'].append(request_item('Complaint: Assign Complaint', 'PUT', '/api/complaints/assign', auth=True, body={
    'complaintId': 1001,
    'assigneeId': 3003
}, description='Assign complaint to a vendor admin or manager.'))
collection['item'].append(request_item('Complaint: Reject Complaint', 'PUT', '/api/complaints/reject', auth=True, body={
    'complaintId': 1001,
    'reason': 'Not a valid fleet complaint'
}, description='Reject a complaint with a reason.'))
collection['item'].append(request_item('Complaint: Make Decision', 'PUT', '/api/complaints/decision', auth=True, body={
    'complaintId': 1001,
    'decision': 'ESCALATE'
}, description='Submit a managerial decision on a complaint.'))
collection['item'].append(request_item('Complaint: List Vendors', 'GET', '/api/complaints/vendors', auth=True, description='Retrieve available vendors.'))
collection['item'].append(request_item('Complaint: Vendor Details', 'POST', '/api/complaints/vendors/details', auth=True, body={
    'vendorId': 5001
}, description='Fetch vendor details for a selected vendor.'))
collection['item'].append(request_item('Complaint: Nearby Vendors', 'POST', '/api/complaints/1001/nearby-vendors', auth=True, body={
    'radiusKm': 10
}, description='Search nearby vendors for a specific complaint.'))
collection['item'].append(request_item('Complaint: Reassign Complaint', 'PUT', '/api/complaints/reassign', auth=True, body={
    'complaintId': 1001,
    'newAssigneeId': 3004
}, description='Reassign a complaint to a different manager or vendor admin.'))

collection['item'].append(request_item('Workflow: Driver Response', 'POST', '/api/workflow/user-response', auth=True, body={
    'taskId': 'task-101',
    'comment': 'Driver inspection complete, issue confirmed'
}, description='Driver responds to an active workflow task.'))
collection['item'].append(request_item('Workflow: Vendor Response', 'POST', '/api/workflow/vendor-response', auth=True, body={
    'taskId': 'task-201',
    'vendorStatus': 'ACCEPTED',
    'notes': 'Vendor will repair within 24 hours'
}, description='Vendor admin responds to a workflow task.'))
collection['item'].append(request_item('Workflow: Manager Response', 'POST', '/api/workflow/manager-response', auth=True, body={
    'taskId': 'task-301',
    'decision': 'APPROVE',
    'notes': 'Move forward with vendor repair'
}, description='Manager approves or rejects a manager-level workflow task.'))

collection['item'].append(request_item('Vendor: List Vendors', 'GET', '/api/vendors/available', auth=True, description='Get available vendors.'))
collection['item'].append(request_item('Vendor: Create Vendor Details', 'POST', '/api/vendors/details', auth=True, body={
    'name': 'ElectroFleet Repair',
    'serviceArea': 'North Sector',
    'specialty': 'Battery and braking systems'
}, description='Create or update vendor profile details.'))
collection['item'].append(request_item('Vendor: Update Availability', 'POST', '/api/vendors/availability', auth=True, body={
    'vendorId': 5001,
    'available': True
}, description='Set vendor availability.'))
collection['item'].append(request_item('Vendor: Search by Expertise', 'POST', '/api/vendors/expertise', auth=True, body={
    'expertise': 'BRAKES'
}, description='Search vendors by expertise area.'))

collection['item'].append(request_item('Admin: List Organizations', 'GET', '/api/users/organizations', auth=True, description='List user organizations.'))
collection['item'].append(request_item('Admin: List Individuals', 'GET', '/api/users/individuals', auth=True, description='List individual users.'))
collection['item'].append(request_item('Admin: Assign Vehicle', 'PUT', '/api/users/assign-vehicle', auth=True, body={
    'userId': 2002,
    'vehicleId': 1001
}, description='Assign a vehicle to a user.'))
collection['item'].append(request_item('Admin: Rate User', 'PUT', '/api/users/3002/rating', auth=True, body={
    'rating': 4.5
}, description='Submit a rating for a target user.'))
collection['item'].append(request_item('Status: Update User Status', 'POST', '/api/status/update', auth=True, body={
    'userId': 2002,
    'status': 'ACTIVE'
}, description='Update application user status.'))

collection['item'].append(request_item('Audit: Complaint Audit Logs', 'POST', '/api/audit-logs/complaint', auth=True, body={
    'complaintId': 1001
}, description='Query audit logs for a specific complaint.'))
collection['item'].append(request_item('Audit: Complaint History', 'GET', '/api/audit-logs/complaint/1001', auth=True, description='Retrieve audit history for a complaint.'))
collection['item'].append(request_item('Audit: Action Audit Logs', 'POST', '/api/audit-logs/action', auth=True, body={
    'action': 'ASSIGN'
}, description='Fetch audit records by action name.'))
collection['item'].append(request_item('Audit: Vehicles by action', 'GET', '/api/audit-logs/vehicle/1001', auth=True, description='Fetch audit logs for a specific vehicle.'))

collection['item'].append(request_item('AI: Create Query', 'POST', '/api/ai/queries', auth=True, body={
    'userId': 2002,
    'prompt': 'What is the best maintenance plan for battery health?',
    'context': 'EV fleet braking system maintenance'
}, description='Submit an AI query.'))
collection['item'].append(request_item('AI: Generate Response', 'POST', '/api/ai/responses', auth=True, body={
    'queryId': 9001,
    'responseText': 'Recommend replacing brake pads and checking battery cooling systems.'
}, description='Create an AI response for a query.'))
collection['item'].append(request_item('AI: User Queries', 'GET', '/api/ai/queries/user/2002', auth=True, description='Fetch AI queries created by a user.'))
collection['item'].append(request_item('AI: Query Responses', 'GET', '/api/ai/responses/query/9001', auth=True, description='Fetch AI responses for a query.'))

collection['item'].append(request_item('Document: Upload Document', 'POST', '/api/documents/upload', auth=True, headers=[{'key': 'Content-Type', 'value': 'multipart/form-data', 'type': 'text'}], body=None, description='Upload a document for user verification.'))
collection['item'].append(request_item('Document: My Documents', 'GET', '/api/documents/my', auth=True, description='List documents uploaded by the current user.'))
collection['item'].append(request_item('Document: Presign Document', 'GET', '/api/documents/presign/1', auth=True, description='Get a presigned URL for a document.'))
collection['item'].append(request_item('Document: Download PAN', 'GET', '/api/documents/download/pan', auth=True, description='Download the current user PAN document.'))
collection['item'].append(request_item('Document: Download PAN by User', 'GET', '/api/documents/download/pan/2002', auth=True, description='Download PAN document for another user.'))

collection['item'].append(request_item('Profile: Profile Me', 'GET', '/api/profile/me', auth=True, description='Get the current user profile.'))
collection['item'].append(request_item('Profile: Complete Profile', 'POST', '/api/profile/complete', auth=True, headers=[{'key': 'Content-Type', 'value': 'multipart/form-data', 'type': 'text'}], body=None, description='Submit multipart profile completion data.'))

collection['item'].append(request_item('Vehicle: List Vehicles', 'GET', '/api/vehicles', auth=True, description='List all vehicles.'))
collection['item'].append(request_item('Vehicle: Get Vehicle', 'GET', '/api/vehicles/1001', auth=True, description='Get specific vehicle details.'))
collection['item'].append(request_item('Vehicle: Update Vehicle', 'PUT', '/api/vehicles/1001', auth=True, body={
    'vehicleId': 1001,
    'status': 'ACTIVE'
}, description='Update vehicle details.'))
collection['item'].append(request_item('Vehicle: Delete Vehicle', 'DELETE', '/api/vehicles/1001', auth=True, description='Delete a vehicle by ID.'))

collection['item'].append(request_item('Service History: Create/Update', 'PUT', '/api/service-history/4001', auth=True, body={
    'vehicleId': 1001,
    'serviceDate': '2026-05-01',
    'odometerReading': 12500,
    'cost': 250.0,
    'notes': 'Brake inspection and pad replacement'
}, description='Update service history record for a vehicle.'))
collection['item'].append(request_item('Service History: Delete', 'DELETE', '/api/service-history/4001', auth=True, description='Delete a service history record.'))
collection['item'].append(request_item('Service History: By Vehicle', 'GET', '/api/service-history/vehicle/1001', auth=True, description='Get service history for a vehicle.'))
collection['item'].append(request_item('Service History: Vehicle Cost', 'GET', '/api/service-history/vehicle/1001/cost', auth=True, description='Get total service cost for a vehicle.'))
collection['item'].append(request_item('Service History: Vehicle Odometer', 'GET', '/api/service-history/vehicle/1001/odometer', auth=True, description='Get the latest odometer reading for a vehicle.'))

# Write outputs
output_files = {
    'FINAL_POSTMAN_COLLECTION.json': collection,
    'POSTMAN_ENVIRONMENT.json': {
        'name': 'EV Fleet Mobility Environment',
        'values': [
            {'key': 'baseUrl', 'value': 'http://localhost:8080', 'enabled': True},
            {'key': 'authToken', 'value': '', 'enabled': True},
            {'key': 'refreshToken', 'value': '', 'enabled': True}
        ]
    },
    'WORKFLOW_TEST_SEQUENCE.md': """# EV Fleet Mobility Workflow Test Sequence

1. Signup and login as a driver
   - `POST /api/auth/signup`
   - `POST /api/auth/login`
2. Create a complaint
   - `POST /api/complaints`
3. Submit a driver workflow response
   - `POST /api/workflow/user-response`
4. Assign the complaint to a manager or vendor admin
   - `PUT /api/complaints/assign`
5. Vendor accepts and updates status
   - `POST /api/workflow/vendor-response`
   - `PUT /api/complaints/status`
6. Manager decision
   - `POST /api/workflow/manager-response`
7. Resolve the complaint
   - `PUT /api/complaints/resolve`
8. Audit and reporting
   - `POST /api/audit-logs/complaint`
   - `GET /api/audit-logs/complaint/{complaintId}`
"""
}

for filename, content in output_files.items():
    with open(Path(filename), 'w', encoding='utf-8') as f:
        if filename.endswith('.json'):
            json.dump(content, f, indent=2)
        else:
            f.write(content)

print('Output files written:')
for filename in output_files:
    print('-', filename)


Extracted DTO fields from 45 files.
Output files written:
- FINAL_POSTMAN_COLLECTION.json
- POSTMAN_ENVIRONMENT.json
- WORKFLOW_TEST_SEQUENCE.md
